# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maqsoodhussain34911-droid/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**My Selected Lane:** Refresh / Content Opportunity Scoring

**Task Type:** Ranking

**Problem:** The content team has many website pages but limited time to review all of them.

**Goal:** Identify which content pages should be reviewed first.

**Why Ranking?** Ranking creates an ordered list of pages according to their review priority.

**Expected Output:** A ranked list of content pages.

**Action:** The content team can review, refresh, or monitor the selected pages.

**Cost of a Wrong Decision:** The team may waste time reviewing pages that do not need improvement or miss pages that need attention.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My Selected Lane: Refresh / Content Opportunity Scoring

Task Type: Ranking

Problem:
The content team has many website pages but limited time to review all of them.

The goal is to identify which content pages should be reviewed first.

Why Ranking?
Ranking helps create an ordered list of pages according to their review priority.

Decision:
Which content page should the content team review first?

Who will use the output?
Content and SEO team members.

Action:
They can review, refresh, expand, or monitor the selected pages.

Cost of a wrong decision:
The team may waste time reviewing pages that do not need improvement, or miss pages that need attention.

Expected Output:
A ranked list of content pages for human review.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The target proxy is whether a content page has a declining performance trend.

The starter dataset contains a column called trend_direction.

I will create a proxy target:

1 = trend_direction is down
0 = Other trend directions

This target is derived from an existing observed trend label. It is not an independently measured future outcome.

Therefore, the result will be used for decision support and page prioritization, not as proof of future decline.

The columns trend_direction and trend_pct must not be used as model input features because they reveal information used to create the target.

In [1]:
import pandas as pd
from pathlib import Path
import subprocess

# Connect to your public GitHub repository
repo_path = Path("/content/flyrank-ml-internship")

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/maqsoodhussain34911-droid/flyrank-ml-internship.git",
            str(repo_path)
        ],
        check=True
    )

# Load the official starter dataset
data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# Create the target proxy
df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype("string")
    .str.lower()
    .eq("down")
    .astype(int)
)

print("\nTarget distribution:")
display(df["is_declining_proxy"].value_counts())

Dataset loaded successfully!
Dataset shape: (30000, 44)

Target distribution:


,count
is_declining_proxy,
1,16262
0,13738


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My success metric is Precision@50.

It measures how many of the top 50 recommended pages match the declining proxy target.

Formula:

Precision@50 = Relevant pages in the top 50 / 50

Why this metric?

The content team has limited time. It needs the most relevant pages at the top of the priority list.

A higher Precision@50 means more of the first 50 recommended pages match the selected proxy.

The metric must be calculated on a ranked list using a valid evaluation dataset. I will not claim a model performance result before running an evaluation.

In [2]:
# Check the positive target rate
positive_pages = int(df["is_declining_proxy"].sum())
total_pages = len(df)

print("Total pages:", total_pages)
print("Declining proxy pages:", positive_pages)
print("Declining proxy percentage:",
      round((positive_pages / total_pages) * 100, 2), "%")

print("\nEvaluation metric: Precision@50")
print("Formula: Relevant pages in top 50 / 50")

Total pages: 30000
Declining proxy pages: 16262
Declining proxy percentage: 54.21 %

Evaluation metric: Precision@50
Formula: Relevant pages in top 50 / 50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The unit of analysis is one content page.

Each row in the starter dataset represents one pseudonymized content page.

The dataset contains content performance information, including impressions, clicks, sessions, content age, and trend information.

I will use this dataframe to understand the available data and the target proxy.

In [3]:
# Check required columns
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
    "is_declining_proxy"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("One row represents one content page.")
print("Total rows:", len(df))

# Show the real dataframe
display(df[required_columns].head(10))

One row represents one content page.
Total rows: 30000


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,1
6,content_9a34b442b552,client_8722616204,20,1,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,1
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


A fixed rule uses one or two conditions to select content pages.

However, content performance depends on many factors, such as impressions, clicks, sessions, CTR, content age, and engagement.

A single fixed threshold may not understand the relationship between all these factors.

Machine Learning can learn patterns from multiple features and produce a more flexible ranking.

The content team can compare the ML ranking with a simple rule-based baseline using Precision@50.

Machine Learning is not automatically better. Its performance must be tested before real use.

The final recommendations will support human decisions. The system will not automatically update website content.

In [4]:
# Check the available information and possible leakage columns

print("Total available features:", len(df.columns))

print("Target column: is_declining_proxy")

print("\nColumns excluded from model inputs:")
print([
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
])

print("\nReason:")
print("IDs are for identification and grouping.")
print("Trend columns reveal information used to create the target.")

print("\nUnit of analysis: One content page")
print("Output: Ranked content pages for human review")

Total available features: 45
Target column: is_declining_proxy

Columns excluded from model inputs:
['content_id', 'client_id', 'trend_direction', 'trend_pct']

Reason:
IDs are for identification and grouping.
Trend columns reveal information used to create the target.

Unit of analysis: One content page
Output: Ranked content pages for human review


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.